# CausalRank — 从 CSV 到静态 HTML 报告（Phase 4）

本 Notebook 演示 Phase 4 的分发与可视化：

1. **CSV 往返**：从 CSV 读入数据 → 完整分析（干净环境安装后的典型用法）；
2. `result.to_html(path)`：**静态** HTML 报告（无服务器、无 JS、无外部资源、
   零新增依赖，图表为内联 SVG）；
3. `result.plot(kind)`：5 种可单独保存的图（estimates / ranks / balance / ess / overlap）；
4. **可追溯性**：报告里的每个数字都来自 `result` 对象（同一舍入规则），
   渲染器不重新拟合、不重新计算任何统计量；
5. **诚实形态**：质量门控横幅、结构化警告、推断阻断状态、公式、tie 约定、
   因果识别免责声明全部随报告输出——报告不会比 result 对象更“干净”。

> **红线不变**：`inconclusive` 不是相等证据；点排名不是最优概率；
> 诊断不能验证无未观测混杂；仅用于研究。

In [1]:
from pathlib import Path

import pandas as pd

from causalrank import CausalRank
from causalrank.simulation import make_dataset

pd.set_option("display.width", 120)
out_dir = Path("artifacts")
out_dir.mkdir(exist_ok=True)

## 1. CSV 往返：模拟数据落盘再读入

真实用户通常从 CSV/Parquet 开始。先把合成数据写到磁盘再读回，
模拟“干净环境 + 用户自己的文件”的用法。

In [2]:
ds = make_dataset("confounded", n=2000, random_state=2026)
csv_path = out_dir / "demo_data.csv"
ds.df.to_csv(csv_path, index=False)
df = pd.read_csv(csv_path)
print("loaded from", csv_path, "->", df.shape)
df.head(3)

loaded from artifacts\demo_data.csv -> (2000, 6)


,x1,x2,x3,x4,treatment,outcome
0,-0.793122,-0.410037,1.0,b,C,0.0
1,0.240571,-0.435962,0.0,b,A,1.0
2,-1.896326,-0.021720,0.0,a,B,0.0


## 2. 完整分析（估计 + 联合 Bootstrap + 排名推断）

In [3]:
model = CausalRank(
    treatment="treatment",
    outcome="outcome",
    confounders=["x1", "x2", "x3", "x4"],
    cross_fit=3,
    n_bootstrap=80,
    confidence_level=0.95,
    random_state=2026,
)
result = model.fit(df)
print("inference_available:", result.inference_available, "| gate:", result.diagnostics.gate)
result.effects.to_dataframe().round(4)

inference_available: True | gate: caution


,label,n_treated,raw_mean,mu,se,ci_low,ci_high,weight_sum,ess
0,A,754,0.4549,0.4219,0.0182,0.3862,0.4575,2231.1808,584.4989
1,B,646,0.4582,0.5412,0.0303,0.4819,0.6006,2303.3496,252.5057
2,C,600,0.4350,0.3834,0.0210,0.3423,0.4245,2297.7461,394.6472


## 3. 生成静态 HTML 报告

`result.to_html(path)` 返回 HTML 字符串并写盘。报告自包含：
内联 CSS + 内联 SVG，双击即可用浏览器打开，不含脚本与外部资源。

In [4]:
report_path = out_dir / "causalrank_report.html"
html = result.to_html(report_path)
print(f"wrote {report_path} ({len(html):,} bytes)")
print("script tags:", html.lower().count("<script"))
# 可追溯性自检：报告包含 result 的每个 mu 值（同一舍入）
est = result.effects.to_dataframe()
assert all(f"{m:.4f}" in html for m in est["mu"])
assert all(f"{s:.4f}" in html for s in est["se"])
print("traceability check: every mu/se value present in the report ✓")

wrote artifacts\causalrank_report.html (30,933 bytes)
script tags: 0
traceability check: every mu/se value present in the report ✓


## 4. 单独的图：`result.plot(kind)`

5 种图都返回内联 SVG 的 `Figure`（Jupyter 里直接内联显示；`fig.save(path)` 可存
独立 `.svg` 或 `.html`）。

In [5]:
for kind in ("estimates", "ranks", "balance", "ess", "overlap"):
    fig = result.plot(kind)
    p = fig.save(out_dir / f"fig_{kind}.svg")
    print(f"{kind:10s} -> {p} ({len(fig.svg):,} bytes)")
result.plot("estimates")

estimates  -> artifacts\fig_estimates.svg (2,667 bytes)
ranks      -> artifacts\fig_ranks.svg (2,162 bytes)
balance    -> artifacts\fig_balance.svg (3,300 bytes)
ess        -> artifacts\fig_ess.svg (1,986 bytes)
overlap    -> artifacts\fig_overlap.svg (2,761 bytes)


Figure(svg='<svg xmlns="http://www.w3.org/2000/svg" width="760" height="172" viewBox="0 0 760 172" font-family="system-ui, sans-serif" font-size="12" role="img"><rect x="0" y="0" width="760" height="172" fill="#ffffff"/><line x1="130" y1="144" x2="700" y2="144" stroke="#94a3b8" stroke-width="1"/><line x1="130.0" y1="144" x2="130.0" y2="148" stroke="#94a3b8"/><text x="130.0" y="161.0" text-anchor="middle" fill="#64748b" font-size="12" font-weight="400">0.322</text><line x1="272.5" y1="144" x2="272.5" y2="148" stroke="#94a3b8"/><text x="272.5" y="161.0" text-anchor="middle" fill="#64748b" font-size="12" font-weight="400">0.397</text><line x1="415.0" y1="144" x2="415.0" y2="148" stroke="#94a3b8"/><text x="415.0" y="161.0" text-anchor="middle" fill="#64748b" font-size="12" font-weight="400">0.471</text><line x1="557.5" y1="144" x2="557.5" y2="148" stroke="#94a3b8"/><text x="557.5" y="161.0" text-anchor="middle" fill="#64748b" font-size="12" font-weight="400">0.546</text><line x1="700.0" y1="144" x2="700.0" y2="148" stroke="#94a3b8"/><text x="700.0" y="161.0" text-anchor="middle" fill="#64748b" font-size="12" font-weight="400">0.621</text><text x="120.0" y="52.0" text-anchor="end" fill="#0f172a" font-size="12" font-weight="400">A</text><line x1="252.8" y1="48.0" x2="388.5" y2="48.0" stroke="#2563eb" stroke-width="2.5" stroke-linecap="round"/><line x1="252.8" y1="43.0" x2="252.8" y2="53.0" stroke="#2563eb" stroke-width="2.5"/><line x1="388.5" y1="43.0" x2="388.5" y2="53.0" stroke="#2563eb" stroke-width="2.5"/><circle cx="320.7" cy="48.0" r="4.5" fill="#1d4ed8" stroke="#ffffff" stroke-width="1.5"/><text x="120.0" y="88.0" text-anchor="end" fill="#0f172a" font-size="12" font-weight="400">B</text><line x1="434.8" y1="84.0" x2="660.7" y2="84.0" stroke="#2563eb" stroke-width="2.5" stroke-linecap="round"/><line x1="434.8" y1="79.0" x2="434.8" y2="89.0" stroke="#2563eb" stroke-width="2.5"/><line x1="660.7" y1="79.0" x2="660.7" y2="89.0" stroke="#2563eb" stroke-width="2.5"/><circle cx="547.8" cy="84.0" r="4.5" fill="#1d4ed8" stroke="#ffffff" stroke-width="1.5"/><text x="120.0" y="124.0" text-anchor="end" fill="#0f172a" font-size="12" font-weight="400">C</text><line x1="169.3" y1="120.0" x2="325.7" y2="120.0" stroke="#2563eb" stroke-width="2.5" stroke-linecap="round"/><line x1="169.3" y1="115.0" x2="169.3" y2="125.0" stroke="#2563eb" stroke-width="2.5"/><line x1="325.7" y1="115.0" x2="325.7" y2="125.0" stroke="#2563eb" stroke-width="2.5"/><circle cx="247.5" cy="120.0" r="4.5" fill="#1d4ed8" stroke="#ffffff" stroke-width="1.5"/><text x="130.0" y="164.0" text-anchor="start" fill="#64748b" font-size="11" font-weight="400">mu_a = E[Y(a)]</text></svg>', kind='estimates', caption='Point estimates mu_a with 95% confidence intervals (joint bootstrap).')

## 5. 推断未启用 / 被阻断时的诚实报告

报告不会伪造不确定性：`n_bootstrap=None` 或成功率门槛未满足时，
排名/两两区间章节会退化为显式说明（“Not available”），
而不是省略或给出错误数字。

In [6]:
plain = CausalRank(
    treatment="treatment",
    outcome="outcome",
    confounders=["x1", "x2", "x3", "x4"],
    cross_fit=3,
    random_state=2026,
).fit(df)
plain_path = out_dir / "causalrank_report_no_inference.html"
plain_html = plain.to_html(plain_path)
print("wrote", plain_path)
print("mentions blocked/absent inference:", "Not available in this result" in plain_html)

wrote artifacts\causalrank_report_no_inference.html
mentions blocked/absent inference: True


## 6. 边界

1. 报告只是 result 对象的**视图**：它不计算任何新统计量，也不验证识别假设；
2. HTML 里的置信区间/排名结论依赖联合 Bootstrap 协方差与渐近正态近似，
   覆盖率以模拟检验为准（`scripts/run_se_validation.py` / `run_rank_validation.py`）；
3. 质量门控 `pass` 只说明本样本没有触发经验警告，**不**说明因果假设成立；
4. 同一 result 两次渲染逐字节一致（无渲染时刻时钟，时间戳取 `metadata['created_at']`）。